# 🧠 IndicAI — Multilingual Intent Classifier
## Training Notebook

**Pipeline:** `SentenceTransformer` embeddings → `LogisticRegression` classifier

**Languages supported:** English · Hindi · Telugu · Tamil · Hinglish

**Intents:** greeting · complaint · support_request · payment_issue · refund_request · cancel_request · recharge_request · unknown_intent

---
### ▶️ How to run
1. Upload `intents.json` to the Colab session (`/content/intents.json`)
2. **Runtime → Run all**
3. Download `model/classifier.pkl` and `model/label_encoder.pkl`
4. Place them in the `model/` folder of your project
5. Run `streamlit run app.py`

---
## Step 1 — Install Packages
We install only what we need. `sentence-transformers` brings the multilingual encoder.
`scikit-learn` provides `LogisticRegression` and evaluation tools.

In [ ]:
# Install required packages (safe for Colab)
import subprocess, sys

packages = [
    "sentence-transformers>=2.6.0",
    "scikit-learn>=1.4.0",
    "scipy>=1.12.0",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages installed.")

---
## Step 2 — Imports

In [ ]:
import re
import json
import pickle
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
from scipy.stats import entropy as scipy_entropy

from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("✅ Imports OK.")

---
## Step 3 — Configuration
Central place for all tunable settings.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
INTENTS_PATH         = Path("intents.json")          # training data
MODEL_DIR            = Path("model")
CLF_PATH             = MODEL_DIR / "classifier.pkl"  # LogisticRegression
LE_PATH              = MODEL_DIR / "label_encoder.pkl"# LabelEncoder

EMBEDDING_MODEL      = "paraphrase-multilingual-MiniLM-L12-v2"  # ~120 MB, loaded dynamically
BATCH_SIZE           = 64     # safe for Colab free tier
TEST_SIZE            = 0.15   # 15% held out for validation
RANDOM_STATE         = 42
LR_C                 = 5.0    # LogisticRegression regularisation
LR_MAX_ITER          = 2000

# OOS (out-of-scope) thresholds — tune in Step 10
CONFIDENCE_THRESHOLD = 0.52
ENTROPY_THRESHOLD    = 2.10

MODEL_DIR.mkdir(exist_ok=True)
print(f"Model output → {MODEL_DIR.resolve()}")

---
## Step 4 — Garbage Detection

Before training or predicting, inputs are validated.
Garbage examples in the training set are filtered out.  
The same function is used at inference time in `app.py`.

**Detected patterns:**
- Symbol-only text: `@@@@@`, `!#$`
- Digit-only text: `123456`
- Repeated single character: `aaaaaaa`, `zzzzz`
- Emoji-only: `🙂🙂🙂`
- Latin keyboard smash with no vowels: `qwerty`, `asdfgh`

**Safe for:** Telugu, Tamil, Hindi, mixed scripts.

In [ ]:
_VOWELS = set('aeiouAEIOU')


def is_garbage(text: str) -> bool:
    """
    Returns True for invalid / unclassifiable inputs.
    Safe for Telugu / Tamil / Hindi / mixed-language text.

    Rejects: '@@@@@', '123456', 'aaaaaaa', 'qwerty', 'asdasdasd'
    Passes:  'hi', 'namaste', 'vanakkam', 'refund kavali'
    """
    t = text.strip()
    if not t or not t.replace(' ', ''):
        return True
    # No real script letters at all
    if not re.search(r'[a-zA-Z\u0900-\u097F\u0C00-\u0C7F\u0B80-\u0BFF\u0A80-\u0AFF]', t):
        return True
    # Digit only
    if re.match(r'^\d+$', t):
        return True
    # Single char repeated 5+ times
    if re.match(r'^(.)\1{4,}$', t, re.UNICODE):
        return True
    # Must have at least 2 real script letters
    letters = re.sub(
        r'[^a-zA-Z\u0900-\u097F\u0C00-\u0C7F\u0B80-\u0BFF\u0A80-\u0AFF]',
        '', t
    )
    if len(letters) < 2:
        return True
    # Latin-only: low vowel ratio → keyboard smash
    latin = re.sub(r'[^a-zA-Z]', '', t)
    if len(latin) >= 5:
        vowel_ratio = sum(1 for c in latin if c in _VOWELS) / len(latin)
        if vowel_ratio < 0.20:
            return True
    # Repeating n-gram pattern (asdasdasd = 'asd'x3)
    if len(t) >= 6:
        for n in (2, 3):
            if len(t) >= n * 3:
                chunk = t[:n]
                if t == chunk * (len(t) // n) and len(set(chunk)) <= 3:
                    return True
    return False

---
## Step 5 — Load Dataset from `intents.json`

The dataset is stored in a readable JSON file grouped by intent.  
We flatten it into `(text, label)` pairs, filter garbage, and lowercase.

In [ ]:
with open(INTENTS_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

texts, labels = [], []
skipped = 0

for intent, examples in data["training_data"].items():
    for ex in examples:
        ex = ex.strip()
        if not ex:
            skipped += 1
            continue
        # Filter training-set garbage (the unknown_intent class also has
        # legit OOS phrases — only skip pure garbage)
        if intent != "unknown_intent" and is_garbage(ex):
            skipped += 1
            continue
        texts.append(preprocess(ex))
        labels.append(intent)

dist = Counter(labels)
total = sum(dist.values())
print(f"Dataset loaded: {total} samples ({skipped} skipped)\n")
print(f"{'Intent':<22} {'Count':>5}")
print("─" * 30)
for intent, count in sorted(dist.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 2)
    print(f"{intent:<22} {count:>5}  {bar}")

---
## Step 6 — Load Sentence Encoder

`paraphrase-multilingual-MiniLM-L12-v2` is a 12-layer multilingual model  
trained on 50+ languages. It understands semantic similarity across languages,  
so "refund kavali" (Telugu) maps close to "i want a refund" (English).  

The model (~120 MB) is downloaded once and **not stored in the repository**.

In [ ]:
print(f"Loading encoder: {EMBEDDING_MODEL}")
print("(This may take 1–2 min on first run; cached afterward)\n")

encoder = SentenceTransformer(EMBEDDING_MODEL)

print(f"✅ Encoder loaded. Max sequence length: {encoder.max_seq_length}")
print(f"   Embedding dimension: {encoder.get_sentence_embedding_dimension()}")

---
## Step 7 — Generate Embeddings (Batched)

We embed in batches of `BATCH_SIZE=64` to avoid RAM spikes on Colab free tier.  
Embeddings are L2-normalized so cosine similarity equals dot product.

In [ ]:
print(f"Embedding {len(texts)} samples in batches of {BATCH_SIZE}…")

all_embeddings = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    emb   = encoder.encode(
        batch,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=BATCH_SIZE,
    )
    all_embeddings.append(emb)
    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  Batch {i // BATCH_SIZE + 1} / {len(texts) // BATCH_SIZE + 1} done")

X = np.vstack(all_embeddings)
print(f"\n✅ Embeddings shape: {X.shape}")

---
## Step 8 — Encode Labels & Train/Test Split

`LabelEncoder` converts intent strings to integer indices.  
We use `stratify=y` to ensure every intent appears in both train and test.

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(labels)

print("Intent → Index mapping:")
for idx, cls in enumerate(le.classes_):
    print(f"  {idx}  {cls}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"\nTrain: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

---
## Step 9 — Train Logistic Regression Classifier

Why Logistic Regression?  
- Fast to train (< 5 sec)  
- Works well with high-dimensional embeddings  
- `predict_proba` gives calibrated confidence scores  
- `class_weight='balanced'` handles class imbalance automatically

In [ ]:
print("Training LogisticRegression…")

clf = LogisticRegression(
    max_iter   = LR_MAX_ITER,
    C          = LR_C,
    class_weight = "balanced",
    multi_class  = "multinomial",
    solver       = "lbfgs",
    random_state = RANDOM_STATE,
)
clf.fit(X_train, y_train)

print("✅ Training complete.")

---
## Step 10 — Evaluation Metrics

We measure accuracy, macro F1, weighted F1, and per-class precision/recall.

In [ ]:
y_pred = clf.predict(X_test)
names  = list(le.classes_)

acc        = accuracy_score(y_test, y_pred)
f1_macro   = f1_score(y_test, y_pred, average="macro")
f1_weighted= f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy  : {acc:.4f}")
print(f"F1 Macro       : {f1_macro:.4f}")
print(f"F1 Weighted    : {f1_weighted:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=names))

### Cross-validation (5-fold StratifiedKFold)
Provides a more robust accuracy estimate over the full dataset.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)

print(f"5-Fold CV F1-macro scores: {cv_scores.round(4)}")
print(f"Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}")

---
## Step 11 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=list(range(len(names))))

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Test Set")

thresh = cm.max() / 2.0
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, str(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=9)

plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrix.png", dpi=150)
plt.show()
print("Saved → model/confusion_matrix.png")

---
## Step 12 — Confidence Threshold Tuning

We scan confidence thresholds from 0.3 → 0.7 and show:
- How many test samples would be rejected as "unknown"
- Accuracy on remaining (accepted) samples

This helps pick `CONFIDENCE_THRESHOLD` wisely.

In [ ]:
probs_test = clf.predict_proba(X_test)
max_probs  = probs_test.max(axis=1)

thresholds = np.arange(0.30, 0.75, 0.05)
print(f"{'Threshold':>10}  {'Accepted':>9}  {'Rejected':>9}  {'Acc on Accepted':>16}")
print("─" * 55)

for th in thresholds:
    mask     = max_probs >= th
    accepted = mask.sum()
    rejected = (~mask).sum()
    if accepted > 0:
        acc_th = accuracy_score(y_test[mask], y_pred[mask])
    else:
        acc_th = float("nan")
    marker = "  ← current" if abs(th - CONFIDENCE_THRESHOLD) < 0.001 else ""
    print(f"      {th:.2f}  {accepted:>9}  {rejected:>9}  {acc_th:>15.3f}{marker}")

print(f"\nConfigured threshold: {CONFIDENCE_THRESHOLD}")
print("Adjust CONFIDENCE_THRESHOLD in Step 3 if needed.")

---
## Step 13 — Verify Predictions on Sample Inputs

Manual spot-check across languages and edge cases.

In [ ]:
def classify(text: str) -> dict:
    """Full inference pipeline (same logic as app.py)."""
    if is_garbage(text):
        return {"intent": "Unknown / Invalid Input", "confidence": 0.0, "status": "rejected"}

    clean = preprocess(text)
    emb   = encoder.encode([clean], normalize_embeddings=True, show_progress_bar=False)
    probs = clf.predict_proba(emb)[0]
    max_p = float(probs.max())
    ent   = float(scipy_entropy(probs))

    if max_p < CONFIDENCE_THRESHOLD or ent > ENTROPY_THRESHOLD:
        return {"intent": "Intent not recognized confidently", "confidence": round(max_p, 4), "status": "low_conf"}

    pred    = int(probs.argmax())
    intent  = le.classes_[pred]
    return {"intent": intent, "confidence": round(max_p, 4), "status": "ok"}


TEST_CASES = [
    # Expected:  valid intents
    ("hi how are you",                       "greeting"),
    ("namaste",                              "greeting"),
    ("vanakkam",                             "greeting"),
    ("ela unnaru",                           "greeting"),
    ("i want to complain",                   "complaint"),
    ("service kharab hai",                   "complaint"),
    ("complaint cheyyali",                   "complaint"),
    ("service romba mosam",                  "complaint"),
    ("app kaam nahi kar raha",               "support_request"),
    ("help kavali",                          "support_request"),
    ("payment fail ho gaya",                 "payment_issue"),
    ("payment aavaledu",                     "payment_issue"),
    ("refund chahiye",                       "refund_request"),
    ("refund kavali",                        "refund_request"),
    ("cancel my order",                      "cancel_request"),
    ("cancel cheyyali",                      "cancel_request"),
    ("recharge karo",                        "recharge_request"),
    ("recharge cheyyali",                    "recharge_request"),
    # Expected:  rejected / unknown
    ("@@@@@",                                "REJECTED"),
    ("123456",                               "REJECTED"),
    ("aaaaaaa",                              "REJECTED"),
    ("qwerty",                               "REJECTED"),
    ("asdasdasd",                            "REJECTED"),
    ("🙂🙂🙂",                              "REJECTED"),
]

print(f"{'Input':<42} {'Expected':<25} {'Got':<30} {'OK?'}")
print("─" * 105)

passed = 0
for text, expected in TEST_CASES:
    r = classify(text)
    got     = r["intent"]
    is_rej  = r["status"] == "rejected"
    ok      = (expected == "REJECTED" and is_rej) or (expected != "REJECTED" and got == expected)
    mark    = "✅" if ok else "❌"
    if ok:
        passed += 1
    conf_str = f"({r['confidence']:.2%})" if r['confidence'] > 0 else ""
    print(f"{text:<42} {expected:<25} {got:<30} {mark} {conf_str}")

print(f"\nPassed: {passed}/{len(TEST_CASES)}")

---
## Step 14 — Save Models

We save `classifier.pkl` and `label_encoder.pkl` **separately** as required  
by the project structure. The transformer weights are NOT saved here —  
they load dynamically from HuggingFace at runtime.

In [ ]:
# Save LogisticRegression
with open(CLF_PATH, "wb") as f:
    pickle.dump(clf, f)
print(f"✅ Saved → {CLF_PATH}  ({CLF_PATH.stat().st_size / 1024:.1f} KB)")

# Save LabelEncoder
with open(LE_PATH, "wb") as f:
    pickle.dump(le, f)
print(f"✅ Saved → {LE_PATH}   ({LE_PATH.stat().st_size / 1024:.1f} KB)")

# Save training summary JSON
summary = {
    "embedding_model":    EMBEDDING_MODEL,
    "total_samples":      len(texts),
    "intents":            list(le.classes_),
    "test_accuracy":      round(acc, 4),
    "f1_macro":           round(f1_macro, 4),
    "f1_weighted":        round(f1_weighted, 4),
    "cv_f1_mean":         round(float(cv_scores.mean()), 4),
    "cv_f1_std":          round(float(cv_scores.std()), 4),
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "entropy_threshold":  ENTROPY_THRESHOLD,
}
with open(MODEL_DIR / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"✅ Saved → model/training_summary.json")

print("\n" + "═" * 50)
print("  TRAINING COMPLETE")
print(f"  Accuracy  : {acc:.4f}")
print(f"  F1 Macro  : {f1_macro:.4f}")
print(f"  CV F1     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("═" * 50)
print()
print("Next steps:")
print("  1. Download model/classifier.pkl and model/label_encoder.pkl")
print("  2. Place them in your project's model/ folder")
print("  3. Run: streamlit run app.py")

---
## Step 15 — Download Models (Colab only)

Run this cell only on **Google Colab** to download the model files.

In [ ]:
try:
    from google.colab import files
    files.download(str(CLF_PATH))
    files.download(str(LE_PATH))
    print("Download triggered for both model files.")
except ImportError:
    print("Not running in Colab — model files are in the model/ folder.")